# Gradient Circuit Discovery, the Tuned Lens, and Max-Activating Examples

Three workflows on top of the basics from [`08_dla_and_circuits.ipynb`](08_dla_and_circuits.ipynb):

1. **Attribution Patching (AtP)** — score *every* module's patch effect from three model passes.
2. **Edge Attribution Patching (EAP)** — score component → residual-stream *edges*, and drive `find_circuit` with them.
3. **Tuned lens** — train per-layer translators for an unbiased early-layer readout.
4. **Max-activating examples** — scan a corpus for what makes a neuron / SAE feature / head fire.

In [ ]:
import interpkit

model = interpkit.load("gpt2")

clean = "The capital of France is"
corrupted = "The capital of Germany is"

## Attribution Patching: every module in three passes

Exhaustive causal tracing (`model.trace`) runs one forward per module. AtP approximates the same patch effects with one clean forward, one corrupted forward, and one backward — scoring **all** modules at once (Syed et al. 2023; correlation with true effects is typically 0.85–0.95).

It's a *first-order* approximation: use it as the fast first look, then confirm top candidates with `trace` or `patch`.

In [ ]:
atp = model.atp(clean, corrupted, top_k=10)

## Edge Attribution Patching

Where AtP scores modules, EAP scores **edges**: how much each component's clean-vs-corrupted delta matters as it flows into each downstream residual-stream layer. The edge at a component's own layer is its total effect (it equals the AtP score — the residual add makes the gradients identical); deeper edges show how the effect persists down the stream.

`ig_steps=5` switches to EAP-IG: gradients averaged over embeddings interpolated from corrupted toward clean, more faithful when the corrupted point sits in a saturated region. Note the pair must tokenize to the **same length**.

In [ ]:
eap = model.eap(clean, corrupted, top_k_edges=10)

print("strongest component:", eap["nodes"][0]["node"], f"({eap['nodes'][0]['score']:+.3f})")

## EAP-driven circuit discovery

`find_circuit(method="eap")` replaces the per-component ablation sweep with the gradient scores above (a handful of passes instead of ~2× components forwards), then still **verifies the circuit causally**: the excluded components are mean-ablated together and the survival of the clean-vs-corrupted distinction is measured.

With EAP methods, `threshold` is a fraction of the top component's absolute score.

In [ ]:
circuit = model.find_circuit(clean, corrupted, method="eap", threshold=0.3)

## The tuned lens

The raw logit lens projects every layer through the *final* norm + unembedding — biased for early layers, whose basis isn't aligned with the unembedding yet. The tuned lens (Belrose et al. 2023) trains one affine translator per layer so each layer's readout matches the model's own final distribution under KL. The model stays frozen; only `n_layers × (hidden² + hidden)` parameters train.

A freshly created lens has identity translators and reproduces the logit lens exactly — training only moves the readout toward the model's true predictions. A few hundred diverse sentences is plenty for small models (we use a tiny corpus and short run here to keep the notebook quick).

In [ ]:
corpus = [
    "The capital of France is Paris.",
    "Water boils at one hundred degrees Celsius.",
    "The sun rises in the east every morning.",
    "Cats are small domesticated animals.",
    "Rome is the capital of Italy.",
    "Snow is cold and white in the winter.",
    "The stock market opens at nine thirty.",
    "Mountains are often covered in snow.",
]

tuned = model.train_tuned_lens(corpus, steps=40, batch_size=4, max_length=32)

Compare the readouts — early layers under the raw lens predict generic tokens; the tuned readout is calibrated to what the model actually ends up predicting:

In [ ]:
model.lens("The capital of France is", position=-1)
model.lens("The capital of France is", position=-1, kind="tuned", tuned_lens=tuned)

Persist with `model.train_tuned_lens(..., save="lens_dir/")` (safetensors + metadata sidecar, validated against the model on load), then reuse via `model.lens(kind="tuned", tuned_lens="lens_dir/")` or the CLI:

```bash
interpkit train-tuned-lens gpt2 --corpus-file texts.txt --save lens_dir/
interpkit lens gpt2 "The capital of France is" --tuned-lens lens_dir/
```

## Max-activating examples: what does this unit fire on?

Scan a corpus and keep the contexts where one unit fires hardest (streaming top-k — memory stays flat however large the corpus). The peak token is highlighted in its context window.

Three unit modes: `neuron=` (raw activation), `feature=` + `sae=` (SAE feature, reusing the encode path from [`04_sae_features.ipynb`](04_sae_features.ipynb)), and `head=` (the head's pre-projection output norm). HuggingFace datasets work too: `dataset="hf:imdb"` with `max_examples=` (requires `pip install 'interpkit[data]'`).

In [ ]:
texts = [
    "The cat sat on the mat.",
    "Paris is the capital of France.",
    "I love programming in Python.",
    "The weather today is sunny and warm.",
    "Quantum physics is fascinating to study.",
    "She bought three apples at the market.",
    "The stock market crashed yesterday morning.",
    "Mountains are covered in deep snow.",
]

hits = model.max_activating(texts, at="transformer.h.6.mlp", neuron=42, top_k=5)

In [ ]:
# Head mode: which contexts make attention head 2 of layer 6 work hardest?
head_hits = model.max_activating(texts, at="transformer.h.6.attn", head=2, top_k=3)

## CLI equivalents

```bash
interpkit atp gpt2 --clean "The capital of France is" --corrupted "The capital of Germany is"
interpkit eap gpt2 --clean "..." --corrupted "..." --ig-steps 5
interpkit find-circuit gpt2 --clean "..." --corrupted "..." --method eap --threshold 0.3
interpkit maxact gpt2 --at transformer.h.6.mlp --neuron 42 --texts-file corpus.txt
```